# Modèle Hiérarchique Bayésien — Risque de Crédit
## Exercice 9 : Estimation des Probabilités de Défaut (Beta-Binomial & Gamma)

---

**Structure mathématique du modèle hiérarchique :**

$$d_c \mid p_c, n_c \sim \text{Binomiale}(n_c, p_c) \quad \text{[Vraisemblance]}$$

$$p_c \mid \alpha_c, \beta_c \sim \text{Beta}(\alpha_c, \beta_c) \quad \text{[Prior]}$$

$$\alpha_c \sim \text{Gamma}(a_\alpha, b_\alpha) \qquad \beta_c \sim \text{Gamma}(a_\beta, b_\beta) \quad \text{[Hyper-prioris]}$$

**Mise à jour bayésienne (conjugaison Beta-Binomiale) :**

$$p_c \mid d_c, n_c \sim \text{Beta}(\alpha_c + d_c, \;\; \beta_c + n_c - d_c)$$

## 0. Imports et Configuration

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'Cadres':   '#2E86AB',
    'Employés': '#A23B72',
    'Ouvriers': '#F18F01',
    'Artisans': '#C73E1D',
}
np.random.seed(42)
N_SIM = 100_000
print("✅ Imports OK")

## Données de l'exercice

| Catégorie | $\alpha_c$ | $\beta_c$ | $n_c$ (effectif) | $d_c$ (défauts) |
|-----------|-----------|-----------|------------------|----------------|
| Cadres    | 2.5       | 7.5       | 50               | 5              |
| Employés  | 3.0       | 9.0       | 80               | 12             |
| Ouvriers  | 2.0       | 8.0       | 60               | 10             |
| Artisans  | 1.5       | 6.5       | 30               | 4              |

In [ ]:
data = {
    'Cadres':   {'alpha': 2.5, 'beta': 7.5, 'n': 50,  'd': 5 },
    'Employés': {'alpha': 3.0, 'beta': 9.0, 'n': 80,  'd': 12},
    'Ouvriers': {'alpha': 2.0, 'beta': 8.0, 'n': 60,  'd': 10},
    'Artisans': {'alpha': 1.5, 'beta': 6.5, 'n': 30,  'd': 4 },
}
print("✅ Données chargées")

---
## Question 1 — Modèle Hiérarchique Complet

Le modèle se décompose en **trois niveaux** :

| Niveau | Objet | Distribution |
|--------|-------|--------------|
| 3 — Hyper-prior | Paramètres $\alpha_c$, $\beta_c$ | $\text{Gamma}(a_\alpha, b_\alpha)$ et $\text{Gamma}(a_\beta, b_\beta)$ |
| 2 — Prior | Probabilité de défaut $p_c$ | $\text{Beta}(\alpha_c, \beta_c)$ |
| 1 — Vraisemblance | Défauts observés $d_c$ | $\text{Binomiale}(n_c, p_c)$ |

**Moyenne et variance a priori :**

$$\mathbb{E}[p_c] = \frac{\alpha_c}{\alpha_c + \beta_c} \qquad \text{Var}[p_c] = \frac{\mathbb{E}[p_c]\,(1 - \mathbb{E}[p_c])}{\alpha_c + \beta_c + 1}$$

In [ ]:
print(f"{'Catégorie':<12} {'α_c':>6} {'β_c':>6} {'E[p] a priori':>15} {'Std[p] a priori':>16}")
print("-" * 58)
for cat, v in data.items():
    mean_prior = v['alpha'] / (v['alpha'] + v['beta'])
    var_prior  = mean_prior * (1 - mean_prior) / (v['alpha'] + v['beta'] + 1)
    print(f"{cat:<12} {v['alpha']:>6.1f} {v['beta']:>6.1f} {mean_prior:>15.4f} {np.sqrt(var_prior):>16.5f}")

---
## Question 2 — Mise à Jour Bayésienne (Distribution A Posteriori)

Grâce à la **conjugaison Beta-Binomiale**, la mise à jour est exacte et analytique :

$$\text{Prior: } p_c \sim \text{Beta}(\alpha_c, \beta_c)$$

$$\text{Posterior: } p_c \mid d_c, n_c \sim \text{Beta}(\underbrace{\alpha_c + d_c}_{\alpha_c^{\text{post}}}, \underbrace{\beta_c + n_c - d_c}_{\beta_c^{\text{post}}})$$

> **Règle pratique :** chaque nouveau défaut incrémente $\alpha$ de 1 ; chaque remboursement incrémente $\beta$ de 1.

In [ ]:
def compute_posterior(alpha_prior, beta_prior, n, d):
    """Retourne les paramètres et moments de la distribution a posteriori Beta."""
    alpha_post = alpha_prior + d
    beta_post  = beta_prior  + (n - d)
    mean_post  = alpha_post / (alpha_post + beta_post)
    var_post   = (alpha_post * beta_post) / (
                  (alpha_post + beta_post)**2 * (alpha_post + beta_post + 1))
    mode_post  = (alpha_post - 1) / (alpha_post + beta_post - 2) \
                  if (alpha_post > 1 and beta_post > 1) else np.nan
    return alpha_post, beta_post, mean_post, var_post, mode_post

posteriors = {}
print(f"{'Catégorie':<12} {'α_post':>8} {'β_post':>8} {'E[p|data]':>11} {'Std[p|data]':>13} {'Mode':>8}")
print("-" * 65)
for cat, v in data.items():
    ap, bp, mp, vp, mode = compute_posterior(v['alpha'], v['beta'], v['n'], v['d'])
    posteriors[cat] = {
        'alpha_post': ap, 'beta_post': bp,
        'mean': mp, 'std': np.sqrt(vp), 'mode': mode,
        'alpha_prior': v['alpha'], 'beta_prior': v['beta'],
        'n': v['n'], 'd': v['d'],
    }
    print(f"{cat:<12} {ap:>8.2f} {bp:>8.2f} {mp:>11.4f} {np.sqrt(vp):>13.5f} {mode:>8.4f}")

---
## Question 3 — Avantages du Modèle Hiérarchique

1. **Partage d'information (*Partial Pooling*)** : les catégories à petits effectifs (Artisans, $n=30$) empruntent de la force statistique aux autres via les hyper-paramètres Gamma.

2. **Quantification de l'incertitude** : la distribution Beta a posteriori fournit des **intervalles de crédibilité** exploitables pour le provisionnement (IFRS 9, Bâle III).

3. **Mise à jour séquentielle** : le posterior d'aujourd'hui devient le prior de demain — aucune refonte complète du modèle n'est nécessaire.

4. **Régularisation naturelle** : évite le sur-ajustement sur petits échantillons, contrairement au MLE.

5. **Cohérence probabiliste** : traitement unifié de l'incertitude paramétrique — supérieur aux méthodes fréquentistes pour la prise de décision bancaire.

---
## Question 4 — Comparaison A Posteriori : Ouvriers vs Employés

On calcule et visualise les distributions a priori et a posteriori pour les deux catégories, ainsi que leurs intervalles de crédibilité à 95%.

In [ ]:
p_grid = np.linspace(0.001, 0.999, 2000)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Question 4 — Distributions a priori et a posteriori\nOuvriers vs Employés",
             fontsize=14, fontweight='bold')

for ax, cat in zip(axes, ['Employés', 'Ouvriers']):
    v   = data[cat]
    p   = posteriors[cat]
    col = COLORS[cat]

    prior_pdf = stats.beta.pdf(p_grid, v['alpha'], v['beta'])
    post_pdf  = stats.beta.pdf(p_grid, p['alpha_post'], p['beta_post'])
    ci_l = stats.beta.ppf(0.025, p['alpha_post'], p['beta_post'])
    ci_h = stats.beta.ppf(0.975, p['alpha_post'], p['beta_post'])

    ax.plot(p_grid, prior_pdf, '--', color=col, lw=2, alpha=0.7,
            label=f"A priori Beta({v['alpha']:.1f}, {v['beta']:.1f})")
    ax.fill_between(p_grid, post_pdf, alpha=0.25, color=col)
    ax.plot(p_grid, post_pdf, '-', color=col, lw=2.5,
            label=f"A posteriori Beta({p['alpha_post']:.1f}, {p['beta_post']:.1f})")
    ax.axvline(p['mean'], color=col, ls=':', lw=1.8,
               label=f"Moyenne = {p['mean']:.4f}")
    ax.axvspan(ci_l, ci_h, alpha=0.10, color=col,
               label=f"IC 95% [{ci_l:.3f}, {ci_h:.3f}]")

    ax.set_title(f"{cat}  (n={v['n']}, d={v['d']})", fontsize=12, fontweight='bold')
    ax.set_xlabel("Probabilité de défaut $p$", fontsize=11)
    ax.set_ylabel("Densité", fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 0.5)

plt.tight_layout()
plt.savefig("q4_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Résultats comparatifs :")
for cat in ['Employés', 'Ouvriers']:
    p   = posteriors[cat]
    ci_l = stats.beta.ppf(0.025, p['alpha_post'], p['beta_post'])
    ci_h = stats.beta.ppf(0.975, p['alpha_post'], p['beta_post'])
    print(f"  {cat:<12} E[p|data]={p['mean']:.4f}  Std={p['std']:.4f}  IC95%=[{ci_l:.4f}, {ci_h:.4f}]")

---
## Question 5 — Simulation Monte Carlo et Visualisation (Toutes Catégories)

On tire $N = 100\,000$ échantillons des distributions a priori et a posteriori pour chaque catégorie, et on compare les histogrammes avec les densités analytiques.

In [ ]:
sim_prior     = {}
sim_posterior = {}

for cat, v in data.items():
    p = posteriors[cat]
    sim_prior[cat]     = np.random.beta(v['alpha'], v['beta'], N_SIM)
    sim_posterior[cat] = np.random.beta(p['alpha_post'], p['beta_post'], N_SIM)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Question 5 — Distributions a priori vs a posteriori (Monte Carlo, N=100 000)\n"
             "Toutes catégories socio-professionnelles",
             fontsize=14, fontweight='bold')

x = np.linspace(0.001, 0.60, 1000)
for ax, cat in zip(axes.flatten(), data.keys()):
    col = COLORS[cat]
    v   = data[cat]
    p   = posteriors[cat]

    ax.hist(sim_prior[cat],     bins=100, alpha=0.40, color='grey',
            density=True, label='A priori (sim.)')
    ax.hist(sim_posterior[cat], bins=100, alpha=0.55, color=col,
            density=True, label='A posteriori (sim.)')
    ax.plot(x, stats.beta.pdf(x, v['alpha'], v['beta']),
            'k--', lw=1.5, label='A priori (analytique)')
    ax.plot(x, stats.beta.pdf(x, p['alpha_post'], p['beta_post']),
            color=col, lw=2.5, label='A posteriori (analytique)')

    ax.set_title(f"{cat}  (n={v['n']}, d={v['d']})", fontsize=11, fontweight='bold')
    ax.set_xlabel("Probabilité de défaut $p$")
    ax.set_ylabel("Densité")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 0.55)

plt.tight_layout()
plt.savefig("q5_simulation.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Simulation terminée — figure sauvegardée : q5_simulation.png")

---
## Question 6 — Interprétation Financière & Actuarielle

**Expected Credit Loss (IFRS 9) :**

$$\text{ECL}_c = \underbrace{\mathbb{E}[p_c \mid \text{données}]}_{\text{PD bayésienne}} \times \text{LGD}_c \times \text{EAD}_c$$

La distribution a posteriori Beta permet de calculer non seulement l'ECL attendu, mais aussi sa **distribution complète** pour le stress-testing réglementaire.

In [ ]:
print(f"{'Catégorie':<12} {'PD priori':>10} {'PD post.':>10} {'Δ PD':>8} "
      f"{'IC 95% low':>12} {'IC 95% high':>12} {'Risque':>10}")
print("-" * 80)

risk_labels = {'Cadres': 'FAIBLE', 'Employés': 'MODÉRÉ',
               'Ouvriers': 'MODÉRÉ', 'Artisans': 'FAIBLE'}

for cat, v in data.items():
    p    = posteriors[cat]
    pd0  = v['alpha'] / (v['alpha'] + v['beta'])
    pd1  = p['mean']
    ci_l = stats.beta.ppf(0.025, p['alpha_post'], p['beta_post'])
    ci_h = stats.beta.ppf(0.975, p['alpha_post'], p['beta_post'])
    print(f"{cat:<12} {pd0:>10.4f} {pd1:>10.4f} {pd1-pd0:>+8.4f} "
          f"{ci_l:>12.4f} {ci_h:>12.4f} {risk_labels[cat]:>10}")

print("\n💡 Implications :")
print("  • Provisionnement différencié par catégorie (IFRS 9 Stage 2/3)")
print("  • IC 95% utilisable pour le stress-testing prudentiel (Bâle III)")
print("  • Posterior Beta intégrable comme prior dans un scoring logistique")

---
## Questions 7-10 — Test d'Hypothèse Bayésien

$$H_0 : p_{\text{Employés}} \leq p_{\text{Ouvriers}} \qquad \text{vs} \qquad H_1 : p_{\text{Employés}} > p_{\text{Ouvriers}}$$

### Question 8 — Règle de décision bayésienne

On calcule la probabilité a posteriori de $H_1$ :

$$P(H_1 \mid \text{données}) = P(p_{\text{Emp}} > p_{\text{Ouv}} \mid \text{données})$$

**Règle :** On rejette $H_0$ si $P(H_1 \mid \text{données}) > \text{seuil}$ (ici seuil = 0.95)

### Question 9 — Calcul par simulation Monte Carlo

In [ ]:
sim_emp = sim_posterior['Employés']
sim_ouv = sim_posterior['Ouvriers']

p_h1  = np.mean(sim_emp > sim_ouv)
p_h0  = 1 - p_h1
diff  = sim_emp - sim_ouv

print(f"P(p_Employés > p_Ouvriers | données) = {p_h1:.5f}  ({p_h1*100:.2f}%)")
print(f"Basé sur {N_SIM:,} simulations Monte Carlo")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Questions 7-10 — Test Bayésien : Employés vs Ouvriers",
             fontsize=13, fontweight='bold')

# Distribution de la différence
ax = axes[0]
ax.hist(diff, bins=150, density=True, color='#5C6BC0', alpha=0.75, edgecolor='none')
ax.axvline(0, color='red', lw=2, ls='--', label='$H_0$ : différence = 0')
ax.axvspan(0, diff.max(), alpha=0.15, color='green',
           label=f'$P(H_1|\\text{{data}}) = {p_h1:.4f}$')
ax.set_title("Distribution postérieure de $(p_{Emp} - p_{Ouv})$", fontsize=11)
ax.set_xlabel("Différence de probabilité de défaut")
ax.set_ylabel("Densité")
ax.legend(fontsize=10)

# Scatter
ax = axes[1]
colors_scatter = np.where(sim_emp[:5000] > sim_ouv[:5000], '#E53935', '#1E88E5')
ax.scatter(sim_ouv[:5000], sim_emp[:5000], alpha=0.08, s=5, c=colors_scatter)
lim = 0.45
ax.plot([0, lim], [0, lim], 'k--', lw=1.5, label='$p_{Emp} = p_{Ouv}$ ($H_0$)')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("$p_{Ouvriers}$ (simulation)")
ax.set_ylabel("$p_{Employés}$ (simulation)")
ax.set_title("Simulations a posteriori — Rouge : $H_1$ vérifié", fontsize=11)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("q7_10_test.png", dpi=150, bbox_inches='tight')
plt.show()

### Question 10 — Décision au seuil 0.95

In [ ]:
SEUIL = 0.95
decision = "REJET de H₀" if p_h1 > SEUIL else "NON-REJET de H₀"

print(f"P(H₁ | données) = {p_h1:.5f}")
print(f"Seuil de décision = {SEUIL}")
print(f"\n→ Décision : {decision}")
if p_h1 > SEUIL:
    print(f"  Les données supportent H₁ avec {p_h1*100:.2f}% de confiance bayésienne.")
else:
    print(f"  Les données ne permettent pas de conclure que p_Emp > p_Ouv au seuil {SEUIL}.")

---
## Question 11 — Facteur de Bayes $B_{01}$

Avec des prior non-informatifs $P(H_0) = P(H_1) = 0.5$ :

$$B_{01} = \frac{P(H_0 \mid \text{données})}{P(H_1 \mid \text{données})} = \frac{1 - P(H_1 \mid \text{données})}{P(H_1 \mid \text{données})}$$

In [ ]:
B01 = p_h0 / p_h1
B10 = p_h1 / p_h0

print(f"P(H₀ | données) = {p_h0:.5f}")
print(f"P(H₁ | données) = {p_h1:.5f}")
print(f"\nB₀₁ = {B01:.5f}   (en faveur de H₀)")
print(f"B₁₀ = {B10:.5f}   (en faveur de H₁)")

---
## Question 12 — Interprétation selon l'Échelle de Jeffreys

| $B_{01}$ | Interprétation |
|----------|---------------|
| $> 100$ | Evidence décisive pour $H_0$ |
| $30 - 100$ | Evidence très forte pour $H_0$ |
| $10 - 30$ | Evidence forte pour $H_0$ |
| $3 - 10$ | Evidence modérée pour $H_0$ |
| $1 - 3$ | Evidence faible pour $H_0$ |
| $< 1$ | Evidence en faveur de $H_1$ |

> *Si $B_{01} < 1$, on lit $B_{10} = 1/B_{01}$ sur la même échelle pour qualifier l'évidence en faveur de $H_1$.*

In [ ]:
def jeffreys_interpretation(B01_val):
    """Interprétation automatique du Facteur de Bayes selon Jeffreys (1961)."""
    if B01_val >= 1:
        val, hyp = B01_val, "H₀"
    else:
        val, hyp = 1 / B01_val, "H₁"
    if val > 100:  label = "DÉCISIVE"
    elif val > 30: label = "TRÈS FORTE"
    elif val > 10: label = "FORTE"
    elif val > 3:  label = "MODÉRÉE"
    else:          label = "FAIBLE"
    return f"Evidence {label} en faveur de {hyp}"

interpretation = jeffreys_interpretation(B01)

print(f"B₀₁ = {B01:.5f}")
print(f"B₁₀ = {B10:.5f}")
print(f"\n→ {interpretation}")

# Visualisation synthèse finale
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Questions 11-12 — Facteur de Bayes et Récapitulatif",
             fontsize=13, fontweight='bold')

# B01 sur l'échelle log
ax = axes[0]
thresholds = [1/100, 1/30, 1/10, 1/3, 1, 3, 10, 30, 100]
colors_j   = ['#b71c1c','#d32f2f','#e57373','#ffcdd2','#f5f5f5',
               '#c8e6c9','#81c784','#388e3c','#1b5e20']
labels_j   = ['Déc. H₁','TF H₁','Fort H₁','Mod. H₁','—',
               'Mod. H₀','Fort H₀','TF H₀','Déc. H₀']
for th, col_j, lb in zip(thresholds, colors_j, labels_j):
    ax.axhline(th, color=col_j, lw=1.2, ls='--', alpha=0.7)
    ax.text(0.55, th, lb, va='center', fontsize=8, color=col_j)
ax.axhline(B01, color='navy', lw=3, label=f"B₀₁ = {B01:.4f}")
ax.set_yscale('log')
ax.set_ylabel("$B_{01}$ (échelle log)", fontsize=11)
ax.set_title(f"Position de $B_{{01}}$ — {interpretation}", fontsize=10)
ax.set_xticks([]); ax.set_xlim(0, 1)
ax.legend(fontsize=10)

# Récapitulatif distributions a posteriori
ax = axes[1]
x = np.linspace(0.001, 0.55, 1000)
for cat in data.keys():
    p = posteriors[cat]
    ax.plot(x, stats.beta.pdf(x, p['alpha_post'], p['beta_post']),
            color=COLORS[cat], lw=2.5, label=f"{cat} (E={p['mean']:.3f})")
    ax.axvline(p['mean'], color=COLORS[cat], lw=0.8, ls=':')
ax.set_title("Récapitulatif : A posteriori de toutes les catégories", fontsize=11)
ax.set_xlabel("Probabilité de défaut $p$")
ax.set_ylabel("Densité")
ax.legend(fontsize=10)
ax.set_xlim(0, 0.50)

plt.tight_layout()
plt.savefig("q11_12_bayes_factor.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Résumé Exécutif

| Question | Résultat clé |
|----------|--------------|
| **Q1** | Modèle Beta-Binomial hiérarchique à 3 niveaux avec hyper-prioris Gamma |
| **Q2** | Mise à jour exacte : $\text{Beta}(\alpha+d,\; \beta+n-d)$ — conjugaison analytique |
| **Q3** | Avantages : pooling partiel, incertitude, séquentiel, régularisation |
| **Q4** | $\mathbb{E}[p_{Emp}|\text{data}] \approx 0.163$ vs $\mathbb{E}[p_{Ouv}|\text{data}] \approx 0.171$ |
| **Q5** | Simulations Monte Carlo $N=100\,000$ — histogrammes vs densités analytiques |
| **Q6** | PD a posteriori intégrée dans ECL (IFRS 9) et scoring bayésien |
| **Q7** | $H_0: p_{Emp} \leq p_{Ouv}$ vs $H_1: p_{Emp} > p_{Ouv}$ |
| **Q8** | Rejeter $H_0$ si $P(H_1\mid\text{données}) > 0.95$ |
| **Q9** | $P(H_1\mid\text{données})$ calculée par Monte Carlo |
| **Q10** | Décision au seuil 0.95 — voir résultat cellule Q10 |
| **Q11** | $B_{01}$ calculé comme rapport des probabilités a posteriori |
| **Q12** | Interprétation automatique selon l'échelle de Jeffreys (1961) |